# Semantic GenAI for BI — Phase 1 Validation Notebook

This notebook validates the full Phase 1 vertical slice:
1. **Sample data loading** via DuckDB adapter
2. **Staging contract** — field names, types, status mapping, net_val computation
3. **Semantic registry** — governed metrics and dimensions
4. **Deterministic metric queries** — all 3 KPIs with filters and grain
5. **API integration** — FastAPI endpoints end-to-end
6. **Narration service** — feature-flagged LLM narration

---
## 1. Environment Setup

In [1]:
import json
import os
import sys
from datetime import date
from pathlib import Path

# Ensure project root is on the path (notebook lives in notebooks/)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

# Set environment for json_duckdb mode
os.environ["SOURCE_MODE"] = "json_duckdb"
os.environ["RAW_JSON_GLOB"] = str(PROJECT_ROOT / "data" / "raw" / "*.json")
os.environ["ENABLE_NARRATION"] = "false"

print(f"Project root: {PROJECT_ROOT}")
print(f"Python version: {sys.version}")

Project root: /home/user/GenAIBI
Python version: 3.11.14 (main, Oct 10 2025, 08:54:04) [GCC 13.3.0]


---
## 2. Load and Inspect Raw Sample Data

In [2]:
raw_path = PROJECT_ROOT / "data" / "raw" / "margin_transactions.json"
with open(raw_path) as f:
    raw_data = json.load(f)

print(f"Raw records loaded: {len(raw_data)}")
print(f"Raw fields: {list(raw_data[0].keys())}")
print()
for row in raw_data:
    print(f"  {row['id']}  acct={row['acct_id']}  amt={row['mc_amt']:>14,.2f}  coll={row['collateral_val']:>14,.2f}  st_cd={row['st_cd']}")

assert len(raw_data) == 10, "Expected 10 raw records"
print("\n[PASS] Raw data loaded successfully.")

Raw records loaded: 10
Raw fields: ['id', 'acct_id', 'mc_amt', 'collateral_val', 'st_cd', 'event_ts']

  MC-001  acct=ACCT-100  amt=  7,500,000.00  coll=  5,000,000.00  st_cd=4
  MC-002  acct=ACCT-101  amt=  3,200,000.00  coll=  2,800,000.00  st_cd=4
  MC-003  acct=ACCT-102  amt= 12,000,000.00  coll=  8,000,000.00  st_cd=4
  MC-004  acct=ACCT-100  amt=  1,500,000.00  coll=  1,600,000.00  st_cd=5
  MC-005  acct=ACCT-103  amt=  6,000,000.00  coll=  4,500,000.00  st_cd=4
  MC-006  acct=ACCT-104  amt=    900,000.00  coll=    950,000.00  st_cd=0
  MC-007  acct=ACCT-105  amt=  8,200,000.00  coll=  6,100,000.00  st_cd=4
  MC-008  acct=ACCT-101  amt=  4,500,000.00  coll=  4,000,000.00  st_cd=5
  MC-009  acct=ACCT-106  amt= 15,000,000.00  coll= 10,000,000.00  st_cd=4
  MC-010  acct=ACCT-102  amt=  2,000,000.00  coll=  2,100,000.00  st_cd=0

[PASS] Raw data loaded successfully.


---
## 3. DuckDB Adapter — Staging Contract Validation

In [3]:
from src.adapters.duckdb_adapter import DuckDBAdapter

adapter = DuckDBAdapter(json_glob=str(PROJECT_ROOT / "data" / "raw" / "*.json"))
staged = adapter.get_staged_data()

print(f"Staged records: {len(staged)}")
print(f"Canonical fields: {list(staged[0].keys())}")
print()

# Display staged data
print(f"{'call_id':<10} {'account_id':<12} {'margin_usd':>14} {'collateral':>14} {'net_val':>14} {'status':<12} {'event_at'}")
print("-" * 100)
for r in staged:
    print(f"{r['call_id']:<10} {r['account_id']:<12} {float(r['margin_amount_usd']):>14,.2f} {float(r['collateral_amount_usd']):>14,.2f} {float(r['net_val']):>14,.2f} {r['status_name']:<12} {r['event_at']}")

assert len(staged) == 10, "Expected 10 staged records"
print("\n[PASS] Staging produced correct record count.")

Staged records: 10
Canonical fields: ['call_id', 'account_id', 'margin_amount_usd', 'collateral_amount_usd', 'net_val', 'status_name', 'event_at']

call_id    account_id       margin_usd     collateral        net_val status       event_at
----------------------------------------------------------------------------------------------------
MC-001     ACCT-100       7,500,000.00   5,000,000.00   2,500,000.00 CONFIRMED    2025-01-05 10:30:00
MC-002     ACCT-101       3,200,000.00   2,800,000.00     400,000.00 CONFIRMED    2025-01-07 14:15:00
MC-003     ACCT-102      12,000,000.00   8,000,000.00   4,000,000.00 CONFIRMED    2025-01-10 09:00:00
MC-004     ACCT-100       1,500,000.00   1,600,000.00    -100,000.00 PENDING      2025-01-12 11:00:00
MC-005     ACCT-103       6,000,000.00   4,500,000.00   1,500,000.00 CONFIRMED    2025-01-15 16:45:00
MC-006     ACCT-104         900,000.00     950,000.00     -50,000.00 CANCELLED    2025-01-18 08:30:00
MC-007     ACCT-105       8,200,000.00   6,100,0

In [4]:
# Validate canonical field names
expected_fields = {"call_id", "account_id", "margin_amount_usd", "collateral_amount_usd", "net_val", "status_name", "event_at"}
actual_fields = set(staged[0].keys())
assert actual_fields == expected_fields, f"Field mismatch: {actual_fields.symmetric_difference(expected_fields)}"
print("[PASS] All canonical fields present.")

# Validate status mapping
status_counts = {}
for r in staged:
    s = r["status_name"]
    status_counts[s] = status_counts.get(s, 0) + 1

print(f"\nStatus distribution: {status_counts}")
assert status_counts["CONFIRMED"] == 6, f"Expected 6 CONFIRMED, got {status_counts.get('CONFIRMED')}"
assert status_counts["PENDING"] == 2, f"Expected 2 PENDING, got {status_counts.get('PENDING')}"
assert status_counts["CANCELLED"] == 2, f"Expected 2 CANCELLED, got {status_counts.get('CANCELLED')}"
print("[PASS] Status code mapping correct (4→CONFIRMED, 5→PENDING, 0→CANCELLED).")

# Validate net_val = margin_amount_usd - collateral_amount_usd
for r in staged:
    expected_net = float(r["margin_amount_usd"]) - float(r["collateral_amount_usd"])
    assert abs(float(r["net_val"]) - expected_net) < 0.01, f"net_val mismatch for {r['call_id']}"
print("[PASS] net_val = margin_amount_usd - collateral_amount_usd for all records.")

# Validate uniqueness of call_id
ids = [r["call_id"] for r in staged]
assert len(ids) == len(set(ids)), "call_id not unique"
print("[PASS] call_id is unique across all records.")

[PASS] All canonical fields present.

Status distribution: {'CONFIRMED': 6, 'PENDING': 2, 'CANCELLED': 2}
[PASS] Status code mapping correct (4→CONFIRMED, 5→PENDING, 0→CANCELLED).
[PASS] net_val = margin_amount_usd - collateral_amount_usd for all records.
[PASS] call_id is unique across all records.


---
## 4. Semantic Registry Validation

In [5]:
from src.services.semantic_registry import SemanticRegistry

registry = SemanticRegistry()

# List all metrics
metrics = registry.list_metrics()
print("Governed Metrics:")
for m in metrics:
    print(f"  {m.name:<25} {m.aggregation}({m.measure})  filters={m.default_filters}")

assert len(metrics) == 3, f"Expected 3 metrics, got {len(metrics)}"
print("\n[PASS] 3 governed metrics registered.")

# List all dimensions
dims = registry.list_dimensions()
print("\nGoverned Dimensions:")
for d in dims:
    print(f"  {d.name:<15} type={d.type}")

assert len(dims) == 3, f"Expected 3 dimensions, got {len(dims)}"
print("\n[PASS] 3 governed dimensions registered.")

Governed Metrics:
  net_margin_exposure       sum(net_val)  filters={'status_name': 'CONFIRMED'}
  high_risk_breaches        count_distinct(call_id)  filters={'status_name': 'CONFIRMED'}
  total_margin_calls        count_distinct(call_id)  filters={}

[PASS] 3 governed metrics registered.

Governed Dimensions:
  event_at        type=time
  status_name     type=categorical
  account_id      type=categorical

[PASS] 3 governed dimensions registered.


In [6]:
# Validate fail-closed behavior: unknown metric
try:
    registry.get_metric("invented_metric")
    assert False, "Should have raised KeyError"
except KeyError as e:
    print(f"[PASS] Unknown metric correctly rejected: {e}")

# Validate fail-closed behavior: unknown filter dimension
try:
    registry.validate_filters("total_margin_calls", {"bogus_field": "x"})
    assert False, "Should have raised ValueError"
except ValueError as e:
    print(f"[PASS] Unknown filter dimension correctly rejected: {e}")

[PASS] Unknown metric correctly rejected: "Unknown metric: 'invented_metric'. Available: ['net_margin_exposure', 'high_risk_breaches', 'total_margin_calls']"
[PASS] Unknown filter dimension correctly rejected: Unknown filter dimension: 'bogus_field'. Available: ['event_at', 'status_name', 'account_id']


---
## 5. Deterministic Metric Queries

In [7]:
from src.core.enums import Grain
from src.services.query_executor import QueryExecutor

executor = QueryExecutor(adapter)

# --- KPI 1: net_margin_exposure (monthly, Jan 2025) ---
metric_nme = registry.get_metric("net_margin_exposure")
rows_nme = executor.execute_metric(
    metric=metric_nme,
    grain=Grain.MONTH,
    filters={},
    start_date=date(2025, 1, 1),
    end_date=date(2025, 1, 31),
)

print("KPI: net_margin_exposure (CONFIRMED, Jan 2025, monthly)")
print(f"  Period: {rows_nme[0]['period']}")
print(f"  Value:  ${float(rows_nme[0]['value']):,.2f}")

# Manual verification:
# MC-001: 7.5M - 5M = 2.5M  (CONFIRMED)
# MC-002: 3.2M - 2.8M = 0.4M  (CONFIRMED)
# MC-003: 12M - 8M = 4M  (CONFIRMED)
# MC-005: 6M - 4.5M = 1.5M  (CONFIRMED)
# MC-007: 8.2M - 6.1M = 2.1M  (CONFIRMED)
# MC-009: 15M - 10M = 5M  (CONFIRMED)
# Total = 15.5M
assert float(rows_nme[0]["value"]) == 15_500_000.00, f"Expected 15500000, got {rows_nme[0]['value']}"
print("  [PASS] net_margin_exposure = $15,500,000.00")

KPI: net_margin_exposure (CONFIRMED, Jan 2025, monthly)
  Period: 2025-01-01 00:00:00
  Value:  $15,500,000.00
  [PASS] net_margin_exposure = $15,500,000.00


In [8]:
# --- KPI 2: high_risk_breaches (monthly, Jan 2025) ---
metric_hrb = registry.get_metric("high_risk_breaches")
rows_hrb = executor.execute_metric(
    metric=metric_hrb,
    grain=Grain.MONTH,
    filters={},
    start_date=date(2025, 1, 1),
    end_date=date(2025, 1, 31),
)

print("KPI: high_risk_breaches (CONFIRMED, mc_amt > $5M, Jan 2025, monthly)")
print(f"  Period: {rows_hrb[0]['period']}")
print(f"  Value:  {int(rows_hrb[0]['value'])} breaches")

# Manual verification:
# CONFIRMED with mc_amt > 5M: MC-001 (7.5M), MC-003 (12M), MC-005 (6M), MC-007 (8.2M), MC-009 (15M) = 5
assert int(rows_hrb[0]["value"]) == 5, f"Expected 5 breaches, got {rows_hrb[0]['value']}"
print("  [PASS] high_risk_breaches = 5")

KPI: high_risk_breaches (CONFIRMED, mc_amt > $5M, Jan 2025, monthly)
  Period: 2025-01-01 00:00:00
  Value:  5 breaches
  [PASS] high_risk_breaches = 5


In [9]:
# --- KPI 3: total_margin_calls (monthly, Jan 2025) ---
metric_tmc = registry.get_metric("total_margin_calls")
rows_tmc = executor.execute_metric(
    metric=metric_tmc,
    grain=Grain.MONTH,
    filters={},
    start_date=date(2025, 1, 1),
    end_date=date(2025, 1, 31),
)

print("KPI: total_margin_calls (all statuses, Jan 2025, monthly)")
print(f"  Period: {rows_tmc[0]['period']}")
print(f"  Value:  {int(rows_tmc[0]['value'])} calls")

assert int(rows_tmc[0]["value"]) == 10, f"Expected 10 calls, got {rows_tmc[0]['value']}"
print("  [PASS] total_margin_calls = 10")

KPI: total_margin_calls (all statuses, Jan 2025, monthly)
  Period: 2025-01-01 00:00:00
  Value:  10 calls
  [PASS] total_margin_calls = 10


In [10]:
# --- Filter test: total_margin_calls filtered by PENDING ---
rows_pending = executor.execute_metric(
    metric=metric_tmc,
    grain=Grain.MONTH,
    filters={"status_name": "PENDING"},
    start_date=date(2025, 1, 1),
    end_date=date(2025, 1, 31),
)

print("Filter test: total_margin_calls WHERE status_name = PENDING")
print(f"  Value:  {int(rows_pending[0]['value'])} calls")
assert int(rows_pending[0]["value"]) == 2, f"Expected 2 PENDING calls, got {rows_pending[0]['value']}"
print("  [PASS] Correctly filtered to 2 PENDING calls.")

# --- Date range test: first half of January ---
rows_half = executor.execute_metric(
    metric=metric_tmc,
    grain=Grain.MONTH,
    filters={},
    start_date=date(2025, 1, 1),
    end_date=date(2025, 1, 15),
)

print("\nDate range test: total_margin_calls Jan 1-15 only")
print(f"  Value:  {int(rows_half[0]['value'])} calls")
assert int(rows_half[0]["value"]) == 5, f"Expected 5 calls in first half, got {rows_half[0]['value']}"
print("  [PASS] Date range filter correct (5 calls in Jan 1-15).")

# --- Daily grain test ---
rows_daily = executor.execute_metric(
    metric=metric_nme,
    grain=Grain.DAY,
    filters={},
    start_date=date(2025, 1, 1),
    end_date=date(2025, 1, 31),
)

print(f"\nDaily grain test: net_margin_exposure returned {len(rows_daily)} day(s)")
assert len(rows_daily) == 6, f"Expected 6 daily rows for confirmed calls, got {len(rows_daily)}"
print("  [PASS] Daily grain returns 6 distinct days for confirmed calls.")

Filter test: total_margin_calls WHERE status_name = PENDING
  Value:  2 calls
  [PASS] Correctly filtered to 2 PENDING calls.

Date range test: total_margin_calls Jan 1-15 only
  Value:  5 calls
  [PASS] Date range filter correct (5 calls in Jan 1-15).

Daily grain test: net_margin_exposure returned 6 day(s)
  [PASS] Daily grain returns 6 distinct days for confirmed calls.


---
## 6. FastAPI Integration Tests

In [11]:
from fastapi.testclient import TestClient
from src.api.main import create_app

app = create_app()
client = TestClient(app)

# --- GET /health ---
resp = client.get("/health")
assert resp.status_code == 200
health = resp.json()
print(f"GET /health → {health}")
assert health["status"] == "ok"
assert health["version"] == "0.1.0"
print("[PASS] /health endpoint works.")

GET /health → {'status': 'ok', 'version': '0.1.0', 'source_mode': 'json_duckdb'}
[PASS] /health endpoint works.


In [12]:
# --- GET /metrics ---
resp = client.get("/metrics")
assert resp.status_code == 200
metrics_data = resp.json()
print("GET /metrics:")
for m in metrics_data["metrics"]:
    print(f"  {m['name']:<25} {m['aggregation']}({m['measure']})")
assert len(metrics_data["metrics"]) == 3
print("[PASS] /metrics returns 3 governed metrics.")

# --- GET /dimensions ---
resp = client.get("/dimensions")
assert resp.status_code == 200
dims_data = resp.json()
print("\nGET /dimensions:")
for d in dims_data["dimensions"]:
    print(f"  {d['name']:<15} type={d['type']}")
assert len(dims_data["dimensions"]) == 3
print("[PASS] /dimensions returns 3 governed dimensions.")

GET /metrics:
  net_margin_exposure       sum(net_val)
  high_risk_breaches        count_distinct(call_id)
  total_margin_calls        count_distinct(call_id)
[PASS] /metrics returns 3 governed metrics.

GET /dimensions:
  event_at        type=time
  status_name     type=categorical
  account_id      type=categorical
[PASS] /dimensions returns 3 governed dimensions.


In [13]:
# --- POST /query-metric: net_margin_exposure ---
resp = client.post("/query-metric", json={
    "metric": "net_margin_exposure",
    "grain": "month",
    "filters": {},
    "start_date": "2025-01-01",
    "end_date": "2025-01-31",
})
assert resp.status_code == 200
result = resp.json()

print("POST /query-metric (net_margin_exposure):")
print(f"  Metric: {result['metric']}")
print(f"  Grain:  {result['grain']}")
print(f"  Filters: {result['filters']}")
print(f"  Rows:   {result['rows']}")
print(f"  Lineage: source={result['lineage']['source']}, model={result['lineage']['staging_model']}")
print(f"  Narration: {result['narration']}")

assert result["rows"][0]["value"] == 15_500_000.00
assert result["lineage"]["metric_name"] == "net_margin_exposure"
assert result["narration"] is None  # narration disabled
print("\n[PASS] /query-metric returns correct deterministic result with lineage.")

POST /query-metric (net_margin_exposure):
  Metric: net_margin_exposure
  Grain:  month
  Filters: {'status_name': 'CONFIRMED'}
  Rows:   [{'period': '2025-01-01', 'value': 15500000.0}]
  Lineage: source=raw_app_data.margin_transactions, model=stg_margin_calls
  Narration: None

[PASS] /query-metric returns correct deterministic result with lineage.


In [14]:
# --- POST /explain-metric ---
resp = client.post("/explain-metric", json={"metric": "net_margin_exposure"})
assert resp.status_code == 200
explain = resp.json()

print("POST /explain-metric (net_margin_exposure):")
print(f"  Description: {explain['description']}")
print(f"  Formula:     {explain['formula']}")
print(f"  Dimensions:  {explain['dimensions']}")
print(f"  Defaults:    {explain['default_filters']}")
print(f"  Source:      {explain['source_model']}")

assert "sum" in explain["formula"]
assert "net_val" in explain["formula"]
print("\n[PASS] /explain-metric returns governed metric explanation.")

POST /explain-metric (net_margin_exposure):
  Description: Total net margin exposure: sum of net_val for confirmed margin calls.
  Formula:     sum(net_val)
  Dimensions:  ['event_at', 'status_name', 'account_id']
  Defaults:    {'status_name': 'CONFIRMED'}
  Source:      stg_margin_calls

[PASS] /explain-metric returns governed metric explanation.


In [15]:
# --- Error handling: unknown metric ---
resp = client.post("/query-metric", json={"metric": "invented_metric", "grain": "day"})
assert resp.status_code == 404
print(f"Unknown metric → HTTP {resp.status_code}: {resp.json()['detail']}")
print("[PASS] Unknown metric correctly returns 404.")

# --- Error handling: invalid filter ---
resp = client.post("/query-metric", json={
    "metric": "total_margin_calls",
    "grain": "day",
    "filters": {"bogus_field": "x"},
})
assert resp.status_code == 400
print(f"Invalid filter → HTTP {resp.status_code}: {resp.json()['detail']}")
print("[PASS] Invalid filter correctly returns 400.")

Unknown metric → HTTP 404: "Unknown metric: 'invented_metric'. Available: ['net_margin_exposure', 'high_risk_breaches', 'total_margin_calls']"
[PASS] Unknown metric correctly returns 404.


Invalid filter → HTTP 400: Unknown filter dimension: 'bogus_field'. Available: ['event_at', 'status_name', 'account_id']
[PASS] Invalid filter correctly returns 400.


---
## 7. Narration Service (Feature-Flagged)

In [16]:
from src.api.schemas import QueryResultRow
from src.llm.narration_service import NarrationService

# Disabled by default
service_off = NarrationService(enabled=False)
result_off = service_off.narrate("net_margin_exposure", [], {}, "month")
assert result_off is None
print("Narration (disabled): None")
print("[PASS] Narration returns None when feature flag is off.")

# Enabled
service_on = NarrationService(enabled=True)
sample_rows = [
    QueryResultRow(period="2025-01-01", value=15_500_000.00),
]
result_on = service_on.narrate(
    metric_name="net_margin_exposure",
    rows=sample_rows,
    filters={"status_name": "CONFIRMED"},
    grain="month",
)
print(f"\nNarration (enabled):\n  {result_on}")
assert "net_margin_exposure" in result_on
assert "CONFIRMED" in result_on
print("\n[PASS] Narration correctly references metric name and filters.")

Narration (disabled): None
[PASS] Narration returns None when feature flag is off.

Narration (enabled):
  The governed metric 'net_margin_exposure' returned 1 period(s) at month grain with filters [status_name=CONFIRMED]. Aggregate total: 15,500,000.00.

[PASS] Narration correctly references metric name and filters.


---
## 8. Summary — All Validations

In [17]:
adapter.close()

print("=" * 60)
print("  PHASE 1 VALIDATION — ALL CHECKS PASSED")
print("=" * 60)
print()
checks = [
    ("Raw data loading", "10 records from JSON"),
    ("Staging contract", "7 canonical fields, types cast, status mapped"),
    ("net_val computation", "margin_amount_usd - collateral_amount_usd"),
    ("call_id uniqueness", "All IDs unique"),
    ("Semantic registry", "3 metrics, 3 dimensions"),
    ("Fail-closed validation", "Unknown metrics/filters rejected"),
    ("net_margin_exposure", "$15,500,000 (6 confirmed calls)"),
    ("high_risk_breaches", "5 breaches (confirmed, >$5M)"),
    ("total_margin_calls", "10 calls (all statuses)"),
    ("Filter by status", "2 PENDING calls"),
    ("Date range filter", "5 calls in Jan 1-15"),
    ("Daily grain", "6 distinct days for confirmed"),
    ("API /health", "status=ok, version=0.1.0"),
    ("API /metrics", "3 governed metrics"),
    ("API /dimensions", "3 governed dimensions"),
    ("API /query-metric", "Deterministic result with lineage"),
    ("API /explain-metric", "Formula and source model"),
    ("API error handling", "404 for unknown metric, 400 for bad filter"),
    ("Narration disabled", "Returns None"),
    ("Narration enabled", "Stub returns grounded summary"),
]
for check, detail in checks:
    print(f"  [PASS] {check:<30} {detail}")

print(f"\n  Total checks: {len(checks)}")
print("  All passed.")

  PHASE 1 VALIDATION — ALL CHECKS PASSED

  [PASS] Raw data loading               10 records from JSON
  [PASS] Staging contract               7 canonical fields, types cast, status mapped
  [PASS] net_val computation            margin_amount_usd - collateral_amount_usd
  [PASS] call_id uniqueness             All IDs unique
  [PASS] Semantic registry              3 metrics, 3 dimensions
  [PASS] Fail-closed validation         Unknown metrics/filters rejected
  [PASS] net_margin_exposure            $15,500,000 (6 confirmed calls)
  [PASS] high_risk_breaches             5 breaches (confirmed, >$5M)
  [PASS] total_margin_calls             10 calls (all statuses)
  [PASS] Filter by status               2 PENDING calls
  [PASS] Date range filter              5 calls in Jan 1-15
  [PASS] Daily grain                    6 distinct days for confirmed
  [PASS] API /health                    status=ok, version=0.1.0
  [PASS] API /metrics                   3 governed metrics
  [PASS] API /dimensio